In [ ]:
from pathlib import Path
import pandas as pd
from utils.data_cleaning import load_raw_data

In [ ]:
df = load_raw_data("data")[['title', 'maintext']]
df = df.head(5000) # Ensure 5,000 rows limit
df

In [ ]:
from datasets import Dataset

# Format data to include the prompt context (aligned with our generation task)
def format_prompt(row):
    # Notice we append the expected generated output at the end so the model can learn it
    return f"Article: {row['maintext']} \n\nTask: Generate a market-aligned title. \n\nTitle: {row['title']}"

# Create the full text column
df["text"] = df.apply(format_prompt, axis=1)

# Convert the pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df[['text']])

# Split into training and evaluation sets
dataset_splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset_splits["train"]
eval_dataset = dataset_splits["test"]

# Export/Save the dataset to disk
dataset_splits.save_to_disk("cache/financial_dataset")

print("Dataset successfully grouped, split, and saved to 'cache/financial_dataset'")
train_dataset

In [6]:
import os
from huggingface_hub import login

# Option A: Interactive login (Safe for Notebooks)
# This will prompt you for the token and save it to ~/.cache/huggingface/token
login()

# Option B: Environment Variable (Recommended for automated scripts)
# os.environ["HF_TOKEN"] = "your_token_here" 
# login(token=os.environ.get("HF_TOKEN"))

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
import torch

model_id = "HuggingFaceTB/SmolLM3-3B"
max_seq_length = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Fix padding for causal completion
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Enable gradient checkpointing to save VRAM
model.gradient_checkpointing_enable()

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 326/326 [00:15<00:00, 20.90it/s]


trainable params: 30,228,480 || all params: 3,105,327,104 || trainable%: 0.9734


In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

output_dir = "./financial_llm_adapters_tmp"

is_bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# Hyperparameters tuned for ROCm/standard setup
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=10,
    max_steps=100, 
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    bf16=is_bf16_supported, 
    fp16=not is_bf16_supported,
    warmup_ratio=0.03,
    optim="adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, # Passing eval dataset directly
    args=training_args,
)

# Start SFT training
trainer.train()

# Save final adapters and tokenizer
save_path = "./financial_llm_adapters"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"LoRA adapters successfully saved to: {save_path}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing eval dataset: 100%|██████████| 250/250 [00:01<00:00, 205.53 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 128012}.


Step,Training Loss,Validation Loss
20,2.394312,2.372300
40,2.300102,2.294806
60,2.332093,2.255938
80,2.256804,2.235525
100,2.117717,2.230439


LoRA adapters successfully saved to: ./financial_llm_adapters


In [13]:
import pandas as pd

# Extract log history
history = pd.DataFrame(trainer.state.log_history)

# Save to CSV
history.to_csv(f"{save_path}/training_logs.csv", index=False)
print("Logs saved as CSV!")

Logs saved as CSV!


In [16]:
import gc
import torch

# 1. Delete the trainer to wipe optimizer states/gradients from VRAM
if 'trainer' in globals():
    del trainer

# 2. Force Python's garbage collector to find the deleted objects
gc.collect()

# 3. Tell ROCm to release the "Reserved" memory back to the GPU
torch.cuda.empty_cache()

# 4. (Optional) Check how much you recovered
free_vram = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Cleaned up! Free VRAM: {free_vram:.2f} GB")

Cleaned up! Free VRAM: 8.89 GB


In [17]:
import evaluate
from tqdm import tqdm
import torch

# Put model in evaluating mode
model.eval()

# Helper generation setup
def generate_title(text_prompt, max_new_tokens=32):
    # Ensure inputs are moved to GPU and cleared later
    inputs = tokenizer(text_prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,  # Crucial for memory efficiency during generation
            do_sample=False
        )
    
    # Explicitly move outputs to CPU to free GPU registers faster
    decoded = tokenizer.decode(outputs[0].cpu(), skip_special_tokens=True)
    return decoded.split("Title: ")[-1].strip()

# Evaluation loop running on subset to save time
rouge = evaluate.load("rouge")

predictions = []
references = []

sample_eval = eval_dataset.select(range(min(50, len(eval_dataset))))

print("Starting evaluation loop...")
for example in tqdm(sample_eval):
    # Extract input context without the trailing ground truth title
    full_text = example['text']
    # Inside your loop, truncate the prompt to 1024 tokens to save VRAM
    prompt_cut = full_text.split("Title: ")[0]
    prompt_cut = tokenizer.decode(tokenizer.encode(prompt_cut)[:1024]) + "Title: "
    reference = full_text.split("Title: ")[-1].strip()
    
    prediction = generate_title(prompt_cut)
    
    predictions.append(prediction)
    references.append(reference)

results = rouge.compute(predictions=predictions, references=references)
print("Evaluation Results:", results)

Starting evaluation loop...


100%|██████████| 50/50 [03:45<00:00,  4.50s/it]

Evaluation Results: {'rouge1': np.float64(0.2187997356265588), 'rouge2': np.float64(0.07847890383946379), 'rougeL': np.float64(0.18716784495037625), 'rougeLsum': np.float64(0.18770075870890673)}
